# 🍎 YOLOv11 Produce Spoilage Detection System
### Lightweight 6-Class Spoilage Classifier for Fresh & Rotten Apples, Bananas, and Oranges
**Author**: Antigravity AI Pair Programmer  
**Model Architecture**: YOLOv11 Nano Classification (`yolo11n-cls`)  
**Hardware Target**: 8GB RAM Computer (CPU / Memory Optimized)

## Cell 1: Environment & Directory Setup (Drive C Cache Optimization)

In [ ]:
import os
import sys
from pathlib import Path

# Ensure cache and run directories are routed to Drive C (22+ GB free)
c_cache_dir = Path(r"C:\Users\HP\.cache\ultralytics")
c_weights_dir = c_cache_dir / "weights"
c_weights_dir.mkdir(parents=True, exist_ok=True)
c_runs_dir = c_cache_dir / "runs"
c_runs_dir.mkdir(parents=True, exist_ok=True)

os.chdir(str(c_weights_dir))
os.environ["YOLO_CONFIG_DIR"] = str(Path(r"C:\Users\HP\.config\ultralytics"))
os.environ["TORCH_HOME"] = str(Path(r"C:\Users\HP\.cache\torch"))
os.environ["TMPDIR"] = str(Path(r"C:\Users\HP\AppData\Local\Temp"))
os.environ["TEMP"] = str(Path(r"C:\Users\HP\AppData\Local\Temp"))

import torch
import torchvision
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

SETTINGS.update({
    'runs_dir': str(c_runs_dir),
    'datasets_dir': str(c_cache_dir / 'datasets'),
    'weights_dir': str(c_weights_dir)
})

print(f"✅ PyTorch Version: {torch.__version__}")
print(f"✅ OpenCV Version: {cv2.__version__}")
print(f"✅ Working Directory: {os.getcwd()}")

## Cell 2: Dataset Verification & Target Class Analysis

In [2]:
dataset_root = Path(r"d:\hackathon\Fruit-Spoilage\moule2\PerishPredict-Smart-Vision-for-Produce-Spoilage-Detection\dataset1")
train_dir = dataset_root / "train"
test_dir = dataset_root / "test"

class_name_map = {
    "apple": "Fresh Apple 🍏",
    "banana": "Fresh Banana 🍌",
    "orange": "Fresh Orange 🍊",
    "rottenapples": "Rotten Apple 🍎⚠️",
    "rottenbanana": "Rotten Banana 🍌⚠️",
    "rottenoranges": "Rotten Orange 🍊⚠️"
}

print("📊 Dataset Class Summary:")
for folder_name, display_name in class_name_map.items():
    tr_count = len(list((train_dir / folder_name).glob("*.*"))) if (train_dir / folder_name).exists() else 0
    te_count = len(list((test_dir / folder_name).glob("*.*"))) if (test_dir / folder_name).exists() else 0
    print(f"  • {display_name:<20} -> Train: {tr_count:>5} images | Test: {te_count:>5} images")

## Cell 3: Load Pre-trained YOLOv11 Nano Classifier

In [3]:
model_weight_file = str(c_weights_dir / "yolo11n-cls.pt")
print(f"📦 Loading YOLOv11 Nano Weights from: {model_weight_file}")

model = YOLO(model_weight_file)
print("✅ Model loaded successfully into memory.")

## Cell 4: Train YOLOv11 Nano Spoilage Classifier on Dataset 1

In [4]:
output_project = str(c_runs_dir / "classify")

print("🚀 Starting Lightweight YOLOv11 Nano Training...")
results = model.train(
    data=str(dataset_root),
    epochs=5,
    imgsz=224,
    batch=32,
    workers=2,
    device="cpu",
    project=output_project,
    name="yolo11n_spoilage",
    exist_ok=True,
    verbose=True
)

best_weights_path = os.path.join(output_project, "yolo11n_spoilage", "weights", "best.pt")
print(f"🎉 Training complete! Best weights saved at: {best_weights_path}")

## Cell 5: Evaluate Trained YOLOv11 Model on Test Set

In [5]:
# Load trained best model
trained_model = YOLO(best_weights_path)

# Evaluate on test split
metrics = trained_model.val(data=str(dataset_root), split="test", device="cpu")

print("\n📈 Validation Results:")
print(f"  • Top-1 Accuracy: {metrics.top1:.4f} ({metrics.top1*100:.2f}%)")
print(f"  • Top-5 Accuracy: {metrics.top5:.4f} ({metrics.top5*100:.2f}%)")

## Cell 6: Visual Inference Function with Spoilage Overlay & Advice

In [6]:
def classify_produce_image(image_path):
    img = Image.open(image_path).convert("RGB")
    res = trained_model.predict(img, device="cpu")[0]
    
    probs = res.probs
    top1_idx = probs.top1
    top1_conf = float(probs.top1conf.cpu().numpy())
    class_name = trained_model.names[top1_idx]
    
    is_rotten = "rotten" in class_name.lower()
    status_str = "ROTTEN / SPOILED ⚠️" if is_rotten else "FRESH ✅"
    display_label = class_name_map.get(class_name.lower(), class_name)
    
    print(f"🖼️ Image: {os.path.basename(image_path)}")
    print(f"🏷️ Classification: {display_label}")
    print(f"🎯 Confidence: {top1_conf*100:.2f}%")
    print(f"💡 Freshness Status: {status_str}")
    
    # Draw badge overlay
    cv_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    badge_color = (0, 0, 230) if is_rotten else (0, 200, 0)
    
    cv2.rectangle(cv_img, (10, 10), (cv_img.shape[1]-10, 60), (20, 20, 20), -1)
    cv2.rectangle(cv_img, (10, 10), (cv_img.shape[1]-10, 60), badge_color, 2)
    text = f"{display_label} ({top1_conf*100:.1f}%)"
    cv2.putText(cv_img, text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{display_label} - Confidence {top1_conf*100:.1f}%")
    plt.show()
    return class_name, top1_conf

# Test on a sample test image
sample_test_img = list((test_dir / "apple").glob("*.png"))[0]
classify_produce_image(str(sample_test_img))

## Cell 7: Video & Webcam Frame Processing Function

In [7]:
def process_video_frame(frame):
    pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    res = trained_model.predict(pil_img, device="cpu")[0]
    
    probs = res.probs
    top1_idx = probs.top1
    top1_conf = float(probs.top1conf.cpu().numpy())
    class_name = trained_model.names[top1_idx]
    
    is_rotten = "rotten" in class_name.lower()
    badge_color = (0, 0, 230) if is_rotten else (0, 200, 0)
    label_text = f"{class_name.upper()} ({top1_conf*100:.1f}%)"
    
    h, w, _ = frame.shape
    cv2.rectangle(frame, (20, 20), (w - 20, 80), (30, 30, 30), -1)
    cv2.rectangle(frame, (20, 20), (w - 20, 80), badge_color, 3)
    cv2.putText(frame, label_text, (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    return frame, class_name, top1_conf

print("✅ Video frame processor function ready.")

## Cell 8: Launch Interactive Gradio Web Interface

In [8]:
import gradio as gr

def gradio_predict(input_image):
    if input_image is None:
        return None, "No image uploaded."
    
    pil_img = Image.fromarray(input_image)
    res = trained_model.predict(pil_img, device="cpu")[0]
    probs = res.probs
    top1_idx = probs.top1
    top1_conf = float(probs.top1conf.cpu().numpy())
    class_name = trained_model.names[top1_idx]
    
    is_rotten = "rotten" in class_name.lower()
    status_emoji = "⚠️ ROTTEN / SPOILED" if is_rotten else "✅ FRESH"
    display_name = class_name_map.get(class_name.lower(), class_name)
    
    # Recommendations
    if is_rotten:
        advice = "❌ DO NOT CONSUME. Produce shows signs of deterioration, discoloration, or decay. Separate immediately from other produce to prevent mold spread."
    else:
        advice = "✅ SAFE FOR CONSUMPTION. Produce is in fresh condition. Store at recommended refrigeration or room temperature."
        
    output_text = f"### Result: {display_name}\n"
    output_text += f"- **Status**: {status_emoji}\n"
    output_text += f"- **Confidence Score**: {top1_conf*100:.2f}%\n\n"
    output_text += f"**Storage & Action Advice**:\n{advice}"
    
    # Draw overlay
    annotated_frame, _, _ = process_video_frame(cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR))
    output_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
    return output_rgb, output_text

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(label="Upload Produce Image (Apple, Banana, Orange)"),
    outputs=[
        gr.Image(label="Annotated Spoilage Overlay"),
        gr.Markdown(label="Classification & Spoilage Diagnosis")
    ],
    title="🍎 PerishPredict - YOLOv11 Produce Spoilage Detector",
    description="Upload an image of an Apple, Banana, or Orange to detect freshness vs. spoilage in real time."
)

print("🌐 Launching Gradio Web App on local server...")
demo.launch(share=False)